In [6]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv('data.csv', encoding='latin1')



In [8]:
# df.info()
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [9]:
# to get missing values
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [10]:
df.shape

(541909, 8)

In [11]:
# df['CustomerID'].sum()
# df['CustomerID'].nunique()
df['CustomerID'].unique()

array([17850., 13047., 12583., ..., 13298., 14569., 12713.], shape=(4373,))

In [12]:
df=df.dropna(subset=['CustomerID'])
df.isnull().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

In [13]:
df.shape

(406829, 8)

In [14]:
df['CustomerID']=df['CustomerID'].astype(int)
df.info()

<class 'pandas.DataFrame'>
Index: 406829 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    406829 non-null  str    
 1   StockCode    406829 non-null  str    
 2   Description  406829 non-null  str    
 3   Quantity     406829 non-null  int64  
 4   InvoiceDate  406829 non-null  str    
 5   UnitPrice    406829 non-null  float64
 6   CustomerID   406829 non-null  int64  
 7   Country      406829 non-null  str    
dtypes: float64(1), int64(2), str(5)
memory usage: 27.9 MB


In [15]:
df=df.drop_duplicates()

In [16]:
df.duplicated().sum()

np.int64(0)

In [17]:
df.dtypes

# invoice date to datetime

InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID       int64
Country            str
dtype: object

In [18]:
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])


In [19]:
df.dtypes

InvoiceNo                 str
StockCode                 str
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID              int64
Country                   str
dtype: object

In [20]:
df = df[df['Quantity'] > 0]

In [21]:
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

In [22]:
df['TotalPrice']=df['Quantity']*df['UnitPrice']

#### FACT TABLE


#### DIMENSION TABLE


In [23]:
dim_customer=df[['UnitPrice','Country']].drop_duplicates()

In [24]:
dim_product=df[['StockCode','Description']].drop_duplicates()


In [25]:
dim_date=df[['InvoiceDate']].drop_duplicates()
dim_date['year']=dim_date['InvoiceDate'].dt.year
dim_date['Month']=dim_date['InvoiceDate'].dt.month
dim_date['Day']=dim_date['InvoiceDate'].dt.day

In [26]:
fact_sales=df[['InvoiceNo','StockCode','CustomerID','InvoiceDate','Quantity','UnitPrice']]

In [27]:
fact_sales['TotalSales'] = fact_sales['Quantity'] * fact_sales['UnitPrice']

In [28]:
fact_sales.head()

,InvoiceNo,StockCode,CustomerID,InvoiceDate,Quantity,UnitPrice,TotalSales
0,536365,85123A,17850,2010-12-01 08:26:00,6,2.55,15.30
1,536365,71053,17850,2010-12-01 08:26:00,6,3.39,20.34
2,536365,84406B,17850,2010-12-01 08:26:00,8,2.75,22.00
3,536365,84029G,17850,2010-12-01 08:26:00,6,3.39,20.34
4,536365,84029E,17850,2010-12-01 08:26:00,6,3.39,20.34


### save fact sales to csv

In [29]:
fact_sales.to_csv('fact_sales.csv',index=False)

In [30]:
dim_customer.to_csv('dim_customer.csv',index=False)
dim_product.to_csv('dim_product.csv', index=False)
dim_date.to_csv('dim_date.csv', index=False)